# PS6 Problem 3: Spam Filter
** – Introduction to Machine Learning (Spring 2026)**

Train a spam filter using the Apache SpamAssassin public mail corpus.  
Classifiers: Multinomial Naive Bayes, KNN, Logistic Regression.

In [ ]:
import os
import email
import warnings
import numpy as np
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## Steps 1–3: Load and Merge Email File Paths

In [ ]:
DATA_DIR = os.path.dirname(os.path.abspath('spam_filter.ipynb'))

def get_email_paths(folder):
    """Return sorted list of email file paths, excluding 'cmds'."""
    folder_path = os.path.join(DATA_DIR, folder)
    files = [os.path.join(folder_path, f) for f in os.listdir(folder_path)
             if f != 'cmds' and not f.startswith('.')]
    return sorted(files)

ham_paths = get_email_paths('easy_ham') + get_email_paths('hard_ham')
spam_paths = get_email_paths('spam') + get_email_paths('spam_2')

print(f"Ham emails:  {len(ham_paths)}")
print(f"Spam emails: {len(spam_paths)}")
print(f"Total:       {len(ham_paths) + len(spam_paths)}")

## Step 4: Parse Emails

In [ ]:
def parse_email(file_path):
    """Parse an email file and return its subject + body as a single string."""
    with open(file_path, 'rb') as f:
        msg = email.message_from_bytes(f.read())

    subject = msg.get('Subject', '')

    body_parts = []
    if msg.is_multipart():
        for part in msg.walk():
            ctype = part.get_content_type()
            cdisp = str(part.get('Content-Disposition', ''))
            if ctype == 'text/plain' and 'attachment' not in cdisp:
                payload = part.get_payload(decode=True)
                if payload:
                    body_parts.append(payload)
    else:
        payload = msg.get_payload(decode=True)
        if payload:
            body_parts.append(payload)

    # Decode bytes to string
    decoded_parts = []
    for bp in body_parts:
        if isinstance(bp, bytes):
            try:
                decoded_parts.append(bp.decode('utf-8', errors='replace'))
            except Exception:
                decoded_parts.append(bp.decode('latin-1', errors='replace'))
        else:
            decoded_parts.append(str(bp))

    body = ' '.join(decoded_parts)
    body = BeautifulSoup(body, 'html.parser').get_text()

    text = f"{subject} {body}" if subject else body
    return text


def load_emails(paths, label):
    """Parse all emails from a list of paths, returning texts and labels."""
    texts, labels = [], []
    for p in paths:
        try:
            text = parse_email(p)
            if text.strip():
                texts.append(text)
                labels.append(label)
        except Exception:
            pass
    return texts, labels

print("Parsing emails...")
ham_texts, ham_labels = load_emails(ham_paths, 0)
spam_texts, spam_labels = load_emails(spam_paths, 1)

all_texts = ham_texts + spam_texts
all_labels = np.array(ham_labels + spam_labels)

print(f"Successfully parsed: {len(ham_texts)} ham, {len(spam_texts)} spam")

## Step 5: TF-IDF Vectorization

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X = vectorizer.fit_transform(all_texts)
y = all_labels

print(f"TF-IDF matrix shape: {X.shape}")

## Step 6: Train-Test Split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

## Step 7: Train & Evaluate Multiple Classifiers

In [ ]:
classifiers = {
    'Multinomial NB': MultinomialNB(alpha=1.0),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0),
}

results = {}

for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)

    results[name] = {'accuracy': acc, 'f1': f1, 'precision': prec, 'recall': rec, 'model': clf}

    print(f"{name}: Accuracy={acc:.4f}, F1={f1:.4f}, Precision={prec:.4f}, Recall={rec:.4f}")

**Note on KNN performance:** KNN performs poorly here because TF-IDF produces very high-dimensional sparse vectors (10,000 features). In such spaces, Euclidean distances between points become nearly uniform (curse of dimensionality), so the notion of "nearest neighbor" loses meaning. Multinomial NB and Logistic Regression work with feature weights directly and are unaffected by this.

### ROC and Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, clf in classifiers.items():
    if hasattr(clf, 'predict_proba'):
        y_scores = clf.predict_proba(X_test)[:, 1]
    else:
        y_scores = clf.decision_function(X_test)

    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_scores)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

    # Precision-Recall curve
    prec_vals, rec_vals, _ = precision_recall_curve(y_test, y_scores)
    ap = average_precision_score(y_test, y_scores)
    axes[1].plot(rec_vals, prec_vals, label=f'{name} (AP = {ap:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves')
axes[1].legend(loc='lower left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 8: Best Model – Confusion Matrix & Classification Report

In [ ]:
best_name = max(results, key=lambda k: results[k]['f1'])
best_clf = results[best_name]['model']

print(f"Best model: {best_name}\n")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, (split_name, X_split, y_split) in enumerate(
    [('Training', X_train, y_train), ('Test', X_test, y_test)]
):
    y_pred = best_clf.predict(X_split)
    print(f"{split_name} Set:")
    print(classification_report(y_split, y_pred, target_names=['Ham', 'Spam']))

    cm = confusion_matrix(y_split, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Ham', 'Spam'])
    disp.plot(ax=axes[idx], cmap='Blues')
    axes[idx].set_title(f'{best_name} – {split_name} Set')

plt.tight_layout()
plt.show()

## Step 9: Learning Curve

In [ ]:
fractions = np.linspace(0.1, 1.0, 10)
train_accs = []
test_accs = []

for frac in fractions:
    n_samples = int(frac * X_train.shape[0])
    X_sub = X_train[:n_samples]
    y_sub = y_train[:n_samples]

    clf_lc = type(best_clf)(**best_clf.get_params())
    clf_lc.fit(X_sub, y_sub)

    train_accs.append(accuracy_score(y_sub, clf_lc.predict(X_sub)))
    test_accs.append(accuracy_score(y_test, clf_lc.predict(X_test)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(fractions * 100, train_accs, 'o-', label='Training Accuracy')
ax.plot(fractions * 100, test_accs, 's-', label='Test Accuracy')
ax.set_xlabel('Training Set Size (%)')
ax.set_ylabel('Accuracy')
ax.set_title(f'Learning Curve – {best_name}')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0.8, 1.02])
plt.tight_layout()
plt.show()

## Step 10: Top 10 Most Indicative Spam Tokens

In [ ]:
# Train a fresh MultinomialNB with Laplace smoothing for log-ratio calculation
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train, y_train)

# feature_log_prob_ gives log P(token | class) with Laplace smoothing
# Class 0 = ham, Class 1 = spam
log_prob_ham = nb_model.feature_log_prob_[0]
log_prob_spam = nb_model.feature_log_prob_[1]

# log ratio = log P(token | spam) - log P(token | ham)
log_ratios = log_prob_spam - log_prob_ham

feature_names = np.array(vectorizer.get_feature_names_out())
top10_idx = np.argsort(log_ratios)[-10:][::-1]

for rank, idx in enumerate(top10_idx, 1):
    print(f"{rank}. {feature_names[idx]}: {log_ratios[idx]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
top_tokens = feature_names[top10_idx]
top_ratios = log_ratios[top10_idx]
ax.barh(range(9, -1, -1), top_ratios, color='crimson', alpha=0.8)
ax.set_yticks(range(9, -1, -1))
ax.set_yticklabels(top_tokens)
ax.set_xlabel('Log Ratio: log P(token|spam) / P(token|ham)')
ax.set_title('Top 10 Most Indicative Spam Tokens')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()